# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

In [1]:
%pip install langchain langchain-classic langchain-openai langchain-tavily tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.


### Setup

In [5]:
import os
from langchain_classic.agents import initialize_agent, AgentType, Tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
import tiktoken
from IPython.display import Markdown, display
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing from .env"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY missing from .env"

tavily_search = TavilySearch(max_results=5)

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4o", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4o")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    try:
        raw = tavily_search.invoke({"query": query})
    except Exception as e:
        # Return the error as text so the agent can retry instead of crashing
        return f"Search failed: {e}. Try a different or broader query."

    # TavilySearch returns a dict: {"results": [{"title", "url", "content", ...}, ...]}
    if isinstance(raw, dict) and raw.get("results"):
        result_text = "\n\n".join(
            f"Title: {r.get('title', '')}\nURL: {r.get('url', '')}\nContent: {r.get('content', '')}"
            for r in raw["results"]
        )
    else:
        result_text = str(raw)

    tokens = encoding.encode(result_text)
    return encoding.decode(tokens[:3500])   # cap tool output so the context stays manageable


# LangChain tool
tools = [Tool(
    name="TavilySafeSearch",
    func=safe_search,
    description=(
        "Searches the web for current, real-time information. Use it for recent news, "
        "events, products, prices, or anything that may have changed after your training data. "
        "Input should be a short, specific search query."
    ),
)]

from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research assistant. Use the TavilySafeSearch tool to find current "
               "information, and base your answers only on the search results."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),   # where the agent's tool calls and results go
])

agent = AgentExecutor(
    agent=create_tool_calling_agent(llm, tools, agent_prompt),
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=8,
)


### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [6]:
# Your prompt here
prompt_1 = """
Use the TavilySafeSearch tool to research the most important advancements in generative AI
applied to healthcare during 2025. Do NOT rely on your prior knowledge: every point must come
from your search results. Run several searches if needed (e.g. clinical documentation,
drug discovery, medical imaging, regulation/FDA approvals).

Format your final answer in Markdown exactly like this:

## Generative AI in Healthcare — 2025 Highlights

### Key Advancements
For each of 4–6 advancements:
- **Name / organisation** — one-sentence description of what it does and why it matters. ([source](URL))

### Main Application Areas
A short bullet list grouping the advancements above by area.

### Challenges & Risks
2–3 bullets (e.g. regulation, safety, bias), each with a source.

### Summary
Two sentences on the overall trend.

Only include information you found in the search results, and include a source URL for every advancement.
"""

# Run it
response_1 = agent.invoke({"input": prompt_1})["output"]



> Entering new AgentExecutor chain...

Invoking: `TavilySafeSearch` with `generative AI healthcare advancements 2025 clinical documentation`


Title: AI in Healthcare 2025 Statistics: Market Size, Adoption, Impact
URL: https://ventionteams.com/healthtech/ai/statistics
Content: ### AI in healthcare adoption rates

In 2025, generative AI continues to gain momentum across healthcare. What began as cautious experimentation in 2024 has turned into a wave of practical applications, from automating clinical documentation to accelerating drug discovery and enhancing patient communication. [...] Among physicians, 68% report an increased use of AI for clinical documentation, striving for accurate recording and efficient management of patient information, while 36% note growth in administrative applications, such as scheduling, claims processing, and revenue cycle management. [...] In 2025, 66% of life sciences executives report investing in generative AI to accelerate research and drug discove

In [7]:
display(Markdown(response_1))


## Generative AI in Healthcare — 2025 Highlights

### Key Advancements
- **Kaiser Permanente / Abridge** — Deployed Abridge’s ambient documentation solution across 40 hospitals and 600+ medical offices, marking the largest generative AI rollout in healthcare history, significantly improving clinical documentation efficiency. ([source](https://menlovc.com/perspective/2025-the-state-of-ai-in-healthcare))
- **Brigham & Women’s Hospital** — Developed a multimodal generative AI model for chest x-ray interpretation, achieving high diagnostic accuracy and streamlining radiologic workflows. ([source](https://www.delveinsight.com/blog/generative-ai-drug-discovery-market-impact))
- **Modella AI / PathChat DX** — Received FDA Breakthrough Device Designation for a generative AI co-pilot in pathology, enhancing diagnostic accuracy and efficiency. ([source](https://www.delveinsight.com/blog/generative-ai-drug-discovery-market-impact))
- **Insilico Medicine** — Achieved positive results from Phase 1 trials of AI-designed drugs for inflammatory bowel disease and mesothelioma, showcasing AI's potential in drug discovery. ([source](https://empowerswiss.org/en/blog/ai-drug-discovery-breakthroughs-move-from-promise-to-proof-in-2025))
- **Hyperfine** — Received CE Mark and UKCA approval for Optive AI™ Software, enhancing portable MRI imaging quality and efficiency. ([source](https://healthmanagement.org/c/healthmanagement/issuearticle/redefining-medical-imaging-through-ai-enabled-precision-and-prediction))

### Main Application Areas
- **Clinical Documentation**: Kaiser Permanente / Abridge
- **Medical Imaging**: Brigham & Women’s Hospital, Hyperfine
- **Pathology**: Modella AI / PathChat DX
- **Drug Discovery**: Insilico Medicine

### Challenges & Risks
- **Regulation**: The FDA is actively exploring regulatory frameworks for generative AI-enabled devices, highlighting the need for clear guidelines. ([source](https://www.sidley.com/en/insights/newsupdates/2025/11/us-fda-and-cms-actions-on-generative-ai-enabled-mental-health-devices-yield-insights-across-ai))
- **Safety**: Ensuring the real-world performance of AI-enabled medical devices remains a critical concern for regulators. ([source](https://www.venable.com/insights/publications/2025/12/the-mind-the-machine-and-the-model-drift-fdas))
- **Bias**: Addressing potential biases in AI models is essential to ensure equitable healthcare outcomes. ([source](https://jmai.amegroups.org/article/view/10846/html))

### Summary
In 2025, generative AI in healthcare has made significant strides, particularly in clinical documentation, medical imaging, and drug discovery. The trend indicates a growing integration of AI technologies into healthcare systems, with ongoing regulatory efforts to ensure safety and efficacy.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [8]:
# Your prompt here
prompt_2 = """
Use the TavilySafeSearch tool to identify the top emerging AI startups of 2025 and their key
innovations. Base your answer only on search results (for example: funding news, "startups
to watch" lists, product launches). Do not include large established companies such as
Google, Microsoft, Meta or Amazon.

Format your final answer in Markdown exactly like this:

## Emerging AI Startups — 2025

| Company | Focus area | Product highlight | Notable 2025 milestone | Source |
|---|---|---|---|---|
(one row per startup, 6–8 startups; the Source column must contain a URL from your search results)

### Trends
3 bullets summarising patterns you see across these startups (e.g. sectors, types of products).

If a detail (like a funding amount) is not in the search results, write "n/a" instead of guessing.
"""

# Run it
response_2 = agent.invoke({"input": prompt_2})["output"]



> Entering new AgentExecutor chain...

Invoking: `TavilySafeSearch` with `top emerging AI startups 2025`


Title: These are the top 25 most valuable AI startups in 2025 👇 | Yurii Rebryk
URL: https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_
Content: : 💵 AI startups captured ~53 % of all global VC funding in 2025 — the first time in history one sector crossed 50 %. ⸻ 🚀 Top 10 AI Startups of 2025 (by Funding & Valuation) OpenAI — $500 B valuation Anthropic — $13 B raised (~$183 B valuation) xAI (Elon Musk) — $10 B raise Thinking Machines Lab — $2 B seed Lila Sciences — $115 M extension (NVIDIA-backed) AI21 Labs — $300 M Series D Perplexity AI — $500 M Series C ElevenLabs — $180 M Series C ($3.3 B valuation) Hugging Face — $235 M Series D [...] # These are the top 25 most valuable AI startups in 2025 👇 | Yurii Rebryk

Agree & Join LinkedIn

By clicking Continue to join or sign in, you agree to LinkedIn’s User Agreemen

In [9]:
display(Markdown(response_2))


## Emerging AI Startups — 2025

| Company | Focus area | Product highlight | Notable 2025 milestone | Source |
|---|---|---|---|---|
| Thinking Machines Lab | AI research and products | AI solutions for various industries | Raised $2B in funding | [Forbes](https://www.forbes.com/lists/ai50) |
| Lila Sciences | Biotechnology | AI-driven drug discovery | $115M extension backed by NVIDIA | [LinkedIn](https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_) |
| Perplexity AI | AI search | Advanced AI-powered search engine | $500M Series C funding | [LinkedIn](https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_) |
| ElevenLabs | AI voice synthesis | AI-driven voice synthesis technology | $180M Series C funding, strategic Nvidia investment | [LinkedIn](https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_) |
| Groq | AI hardware | Fast AI inference technology | $750M raise, $6.9B valuation | [Medium](https://medium.com/jonathans-musings/ai-startups-you-should-know-48d69d1ddf61) |
| Crusoe | AI infrastructure | AI data center and energy infrastructure | $1.375B Series E funding | [Medium](https://medium.com/jonathans-musings/ai-startups-you-should-know-48d69d1ddf61) |
| Cohere | Enterprise AI | Secure enterprise AI solutions | $100M funding extension, $7B valuation | [LinkedIn](https://www.linkedin.com/posts/yrebryk_these-are-the-top-25-most-valuable-ai-startups-activity-7387111846970908672-kPS_) |

### Trends
- **Enterprise AI Solutions**: Many startups are focusing on providing secure and scalable AI solutions for enterprises, indicating a strong demand for AI in business applications.
- **AI Infrastructure and Hardware**: There is significant investment in AI infrastructure and hardware, such as data centers and inference technology, highlighting the importance of supporting AI's computational needs.
- **Biotechnology and Healthcare**: AI's application in biotechnology and healthcare is gaining traction, with startups like Lila Sciences focusing on drug discovery, showcasing AI's potential in life sciences.

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [10]:
# Your prompt here
prompt_3 = """
Use the TavilySafeSearch tool to compare OpenAI's ChatGPT Team plan and Anthropic's Claude Pro plan.
Search each product separately, and check the official pricing pages and recent reviews.
Note that plan names and prices change often: if a plan has been renamed or replaced,
say so and compare the current equivalent.

Format your final answer in Markdown exactly like this:

## ChatGPT Team vs. Claude Pro

| | ChatGPT Team | Claude Pro |
|---|---|---|
| Price (monthly) | | |
| Target user | | |
| Key features | | |
| Usage limits | | |
| Collaboration / admin features | | |

### What reviewers say
- **ChatGPT Team:** 2–3 points from reviews (with source URLs)
- **Claude Pro:** 2–3 points from reviews (with source URLs)

### Which one fits whom
Two or three sentences, based only on the information above.

Cite a source URL for every price and feature. If something isn't in the search results, write "not found".
"""

# Run it
response_3 = agent.invoke({"input": prompt_3})["output"]



> Entering new AgentExecutor chain...

Invoking: `TavilySafeSearch` with `OpenAI ChatGPT Team plan pricing and reviews`


Title: ChatGPT Pricing Guide: Plans, Token Governance & Spend Control | CloudEagle.ai
URL: https://www.cloudeagle.ai/blogs/blog-chatgpt-pricing-guide
Content: ChatGPT pricing in 2026 runs from $0 (Free) to $500/month (Pro) for individuals, with business seats at $20 to $125 per user and custom Enterprise contracts.
 OpenAI changed its plans this year: Team became Business, a cheaper Go plan launched, and Pro now comes in three price tiers.
 Pro plans give heavy users more usage, the most advanced models, and an AI assistant that works in the background. New $200 Pro subscribers now get less usage for the same price.

Title: ChatGPT Enterprise Pricing: Cost, Plans, and Fees (2026)
URL: https://www.gosearch.ai/faqs/chatgpt-enterprise-pricing-explained-cost-tiers-hidden-fees-gosearch-comparison
Content: ChatGPT Business (formerly ChatGPT Team) is OpenAI’s self-serve 

In [11]:
display(Markdown(response_3))


## ChatGPT Team vs. Claude Pro

| | ChatGPT Team | Claude Pro |
|---|---|---|
| Price (monthly) | $25 per user ([source](https://help.openai.com/en/articles/8792828-chatgpt-business-overview)) | $20 ([source](https://www.anthropic.com/news/claude-pro)) |
| Target user | Organizations needing admin controls and data protections ([source](https://www.layer3labs.io/guides/chatgpt-pricing)) | Individuals needing more usage than the free tier ([source](https://coursiv.io/blog/is-claude-pro-worth-it)) |
| Key features | Access to ChatGPT Work and Codex, admin controls ([source](https://help.openai.com/en/articles/8792828-chatgpt-business-overview)) | 5x more usage of Claude 2, access to Claude Code and Cowork ([source](https://www.anthropic.com/news/claude-pro)) |
| Usage limits | Fixed per-user cost with flexible pricing for additional usage ([source](https://help.openai.com/en/articles/8792828-chatgpt-business-overview)) | Approximately 44,000 tokens per 5-hour period ([source](https://www.jetadmin.io/blog/claude-code-pricing-plans-api-rates-and-total-cost-in-2026)) |
| Collaboration / admin features | Admin controls, data protections ([source](https://www.layer3labs.io/guides/chatgpt-pricing)) | Not found |

### What reviewers say
- **ChatGPT Team:** 
  - Offers a range of plans suitable for different productivity needs ([source](https://tldv.io/blog/chatgpt-pricing)).
  - The Business plan is ideal for organizations needing admin controls ([source](https://www.layer3labs.io/guides/chatgpt-pricing)).
- **Claude Pro:** 
  - Worth it for users who regularly hit free-plan limits ([source](https://coursiv.io/blog/is-claude-pro-worth-it)).
  - Provides significant usage increase over free tier ([source](https://www.youtube.com/watch?v=Sdewx3GpJ1Q)).

### Which one fits whom
ChatGPT Team (now Business) is best for organizations that require admin controls and data protection, making it suitable for structured team environments. Claude Pro is more suited for individual users who need more extensive usage than the free tier allows, particularly those who frequently use Claude for coding and collaborative tasks.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
